# 📊 Elden Ring — Chẩn Đoán Mô Hình OLS (Diagnostics)

**Mục tiêu**: Thực hiện chiến lược kiểm định **In-sample** và **Out-of-sample** cho 3 mô hình Dynamic OLS.

**Nội dung**:
1. Tải dữ liệu Train/Test và 3 mô hình đã huấn luyện
2. Kiểm tra đa cộng tuyến (VIF) cho Model 3
3. Kiểm định In-sample Model 1 (P-value, AIC/BIC, Durbin-Watson)
4. Chẩn đoán dấu hệ số Model 3
5. Kiểm định Out-of-sample (MAPE trên tập Test)

---
## Cell 1 — Tải Dữ Liệu và Mô Hình

In [1]:
"""
Cell 1: Tải tập Train, tập Test và load 3 mô hình OLS đã lưu.
Thiết lập logging thay cho print() theo quy tắc production.
"""

import logging
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.linear_model import RegressionResultsWrapper
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

# --- Cấu hình logging (hiển thị trong notebook output) ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger = logging.getLogger("diagnostics")

# --- Hằng số đường dẫn (pathlib) ---
PROJECT_ROOT: Path = Path.cwd().parent  # notebooks/ -> project root
DATA_DIR: Path = PROJECT_ROOT / "data" / "processed"
MODEL_DIR: Path = PROJECT_ROOT / "models"

# --- Tải tập Train ---
train_df: pd.DataFrame = pd.read_csv(
    DATA_DIR / "EldenRing_Train.csv", parse_dates=["Date"], index_col="Date"
)
logger.info("Tập Train: %d dòng, %d cột", *train_df.shape)

# --- Tải tập Test ---
test_df: pd.DataFrame = pd.read_csv(
    DATA_DIR / "EldenRing_Test.csv", parse_dates=["Date"], index_col="Date"
)
logger.info("Tập Test: %d dòng, %d cột", *test_df.shape)

# --- Tải 3 mô hình OLS ---
def load_model(filepath: Path) -> RegressionResultsWrapper:
    """Tải mô hình OLS từ file .pkl."""
    try:
        with open(filepath, "rb") as f:
            model = pickle.load(f)
        logger.info("Đã tải mô hình: %s", filepath.name)
        return model
    except FileNotFoundError as e:
        logger.error("Không tìm thấy file mô hình: %s", filepath)
        raise

model_1: RegressionResultsWrapper = load_model(MODEL_DIR / "EldenRing_Model1_Base.pkl")
model_2: RegressionResultsWrapper = load_model(MODEL_DIR / "EldenRing_Model2_HypeContent.pkl")
model_3: RegressionResultsWrapper = load_model(MODEL_DIR / "EldenRing_Model3_Interaction.pkl")

logger.info("✅ Tải thành công 3 mô hình và dữ liệu Train/Test")

2026-04-08 22:01:35,097 | INFO | Tập Train: 1399 dòng, 15 cột


2026-04-08 22:01:35,107 | INFO | Tập Test: 90 dòng, 15 cột


2026-04-08 22:01:35,117 | INFO | Đã tải mô hình: EldenRing_Model1_Base.pkl


2026-04-08 22:01:35,123 | INFO | Đã tải mô hình: EldenRing_Model2_HypeContent.pkl


2026-04-08 22:01:35,130 | INFO | Đã tải mô hình: EldenRing_Model3_Interaction.pkl


2026-04-08 22:01:35,131 | INFO | ✅ Tải thành công 3 mô hình và dữ liệu Train/Test


---
## Cell 2 — Kiểm Tra Đa Cộng Tuyến (VIF) cho Model 3

In [2]:
"""
Cell 2: Tính chỉ số VIF (Variance Inflation Factor) cho tập Train
của Model 3 để kiểm tra đa cộng tuyến.

Ngưỡng chẩn đoán:
    - VIF < 5 : Không có đa cộng tuyến đáng kể
    - 5 ≤ VIF < 10 : Đa cộng tuyến trung bình, cần theo dõi
    - VIF ≥ 10 : Đa cộng tuyến nghiêm trọng, cần xử lý
"""


def compute_vif(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """Tính VIF cho danh sách biến độc lập.

    Args:
        df: DataFrame chứa dữ liệu.
        feature_cols: Danh sách tên cột biến độc lập.

    Returns:
        DataFrame chứa tên biến và VIF tương ứng.
    """
    try:
        # Thêm hằng số để tính VIF chính xác — has_constant='add' bắt buộc thêm cột const
        X: pd.DataFrame = sm.add_constant(df[feature_cols], has_constant='add')

        vif_data: list[dict[str, object]] = []
        for i in range(X.shape[1]):
            vif_value: float = variance_inflation_factor(X.values, i)
            vif_data.append({
                "Biến": X.columns[i],
                "VIF": round(vif_value, 4),
            })

        vif_df: pd.DataFrame = pd.DataFrame(vif_data)
        return vif_df

    except Exception as e:
        logger.error("LỖI khi tính VIF: %s", e)
        raise


# --- Biến độc lập của Model 3 ---
model_3_features: list[str] = [
    "Lag_Player", "Lag_7_Player", "Discount_Ratio", "Is_Weekend",
    "Years_Since_Release", "Log_Twitch_Avg", "Trend_Index",
    "Is_Major_Update", "Is_Minor_Update",
    "Interaction_Discount_Time", "Interaction_Update_Time",
]

vif_result: pd.DataFrame = compute_vif(train_df, model_3_features)

logger.info("=" * 50)
logger.info("BẢNG VIF — MODEL 3 (Interaction)")
logger.info("=" * 50)
for _, row in vif_result.iterrows():
    # Chẩn đoán tự động cho từng biến
    status: str = "✅ OK"
    if row["VIF"] >= 10:
        status = "🔴 NGHIÊM TRỌNG"
    elif row["VIF"] >= 5:
        status = "🟡 TRUNG BÌNH"
    logger.info("  %-30s VIF = %8.4f  %s", row["Biến"], row["VIF"], status)

# Cảnh báo tổng thể
high_vif = vif_result[vif_result["VIF"] >= 10]
if not high_vif.empty:
    logger.warning(
        "⚠️ CẢNH BÁO: Các biến sau có VIF ≥ 10 (đa cộng tuyến nghiêm trọng): %s",
        high_vif["Biến"].tolist(),
    )
else:
    logger.info("✅ Không có biến nào có VIF ≥ 10 — Đa cộng tuyến trong ngưỡng chấp nhận")

# Hiển thị bảng VIF
vif_result

2026-04-08 22:01:35,416 | INFO | ==================================================


2026-04-08 22:01:35,417 | INFO | BẢNG VIF — MODEL 3 (Interaction)


2026-04-08 22:01:35,418 | INFO | ==================================================


2026-04-08 22:01:35,420 | INFO |   const                          VIF = 439.8983  🔴 NGHIÊM TRỌNG


2026-04-08 22:01:35,420 | INFO |   Lag_Player                     VIF =  16.1992  🔴 NGHIÊM TRỌNG


...

2026-04-08 22:01:35,426 | INFO |   Is_Minor_Update                VIF =   1.0657  ✅ OK


2026-04-08 22:01:35,427 | INFO |   Interaction_Discount_Time      VIF =   8.5957  🟡 TRUNG BÌNH


2026-04-08 22:01:35,428 | INFO |   Interaction_Update_Time        VIF =   2.7467  ✅ OK


2026-04-08 22:01:35,431 | WARNING | ⚠️ CẢNH BÁO: Các biến sau có VIF ≥ 10 (đa cộng tuyến nghiêm trọng): ['const', 'Lag_Player', 'Lag_7_Player']


,Biến,VIF
0,const,439.8983
1,Lag_Player,16.1992
2,Lag_7_Player,15.9253
3,Discount_Ratio,8.8431
4,Is_Weekend,1.1267
5,Years_Since_Release,1.5536
6,Log_Twitch_Avg,1.0551
7,Trend_Index,1.4587
8,Is_Major_Update,2.7001
9,Is_Minor_Update,1.0657


---
## Cell 3 — Kiểm Định In-sample Model 1 (P-value, AIC/BIC, Durbin-Watson)

In [3]:
"""
Cell 3: Kiểm định In-sample cho Model 1 (Base).

Chỉ số kiểm tra:
    - P-value của từng biến độc lập
    - AIC / BIC (so sánh giữa các mô hình)
    - Durbin-Watson (DW): Kiểm tra tự tương quan bậc 1 của phần dư

Quy tắc chẩn đoán DW:
    - DW ≈ 2   : Không có tự tương quan
    - DW < 1.5  : Tự tương quan dương → cần thêm biến lag
    - DW > 2.5  : Tự tương quan âm
"""


def diagnose_in_sample(
    model: RegressionResultsWrapper, model_name: str
) -> dict[str, float]:
    """Chẩn đoán In-sample cho một mô hình OLS.

    Args:
        model: Mô hình OLS đã fit.
        model_name: Tên mô hình (dùng cho logging).

    Returns:
        Dictionary chứa các chỉ số chẩn đoán.
    """
    try:
        # --- P-values ---
        logger.info("=" * 60)
        logger.info("KIỂM ĐỊNH IN-SAMPLE: %s", model_name)
        logger.info("=" * 60)
        logger.info("--- P-values ---")
        for var_name, pval in model.pvalues.items():
            sig: str = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
            logger.info("  %-25s p = %.6f  %s", var_name, pval, sig)

        # --- AIC / BIC ---
        aic: float = model.aic
        bic: float = model.bic
        logger.info("--- Tiêu chí thông tin ---")
        logger.info("  AIC = %.4f", aic)
        logger.info("  BIC = %.4f", bic)

        # --- Durbin-Watson ---
        dw: float = durbin_watson(model.resid)
        logger.info("--- Kiểm định Durbin-Watson ---")
        logger.info("  DW = %.4f", dw)

        # Chẩn đoán tự động DW
        if dw < 1.5:
            logger.warning(
                "⚠️ CẢNH BÁO: DW = %.4f < 1.5 — Mô hình bị tự tương quan dương! "
                "Đề xuất thêm Lag_7_Player vào mô hình để giảm tự tương quan.",
                dw,
            )
        elif dw > 2.5:
            logger.warning(
                "⚠️ CẢNH BÁO: DW = %.4f > 2.5 — Có dấu hiệu tự tương quan âm.",
                dw,
            )
        else:
            logger.info(
                "  ✅ DW = %.4f — Nằm trong khoảng chấp nhận (1.5 - 2.5), "
                "không có bằng chứng tự tương quan nghiêm trọng.",
                dw,
            )

        return {"aic": aic, "bic": bic, "dw": dw}

    except Exception as e:
        logger.error("LỖI khi chẩn đoán In-sample %s: %s", model_name, e)
        raise


# --- Chạy chẩn đoán cho Model 1 ---
diag_m1: dict[str, float] = diagnose_in_sample(model_1, "Model 1 (Base)")

# In summary cho Model 1
logger.info("\n--- BẢNG SUMMARY ĐẦY ĐỦ MODEL 1 ---")
logger.info(model_1.summary().as_text())

2026-04-08 22:01:35,477 | INFO | ============================================================


2026-04-08 22:01:35,477 | INFO | KIỂM ĐỊNH IN-SAMPLE: Model 1 (Base)


2026-04-08 22:01:35,478 | INFO | ============================================================


2026-04-08 22:01:35,479 | INFO | --- P-values ---


2026-04-08 22:01:35,480 | INFO |   const                     p = 0.039814  *


...

2026-04-08 22:01:35,485 | INFO | --- Kiểm định Durbin-Watson ---


2026-04-08 22:01:35,485 | INFO |   DW = 1.1918


2026-04-08 22:01:35,486 | WARNING | ⚠️ CẢNH BÁO: DW = 1.1918 < 1.5 — Mô hình bị tự tương quan dương! Đề xuất thêm Lag_7_Player vào mô hình để giảm tự tương quan.


2026-04-08 22:01:35,486 | INFO | 
--- BẢNG SUMMARY ĐẦY ĐỦ MODEL 1 ---


2026-04-08 22:01:35,508 | INFO |                             OLS Regression Results                            
Dep. Variable:             Log_Player   R-squared:                       0.978
Model:                            OLS   Adj. R-squared:                  0.978
Method:                 Least Squares   F-statistic:                     7786.
Date:                Wed, 08 Apr 2026   Prob (F-statistic):               0.00
Time:                        22:01:35   Log-Likelihood:                 1283.6
No. Observations:                1399   AIC:                            -2555.
Df Residuals:                    1393   BIC:                            -2524.
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
c

---
## Cell 4 — Chẩn Đoán Dấu Hệ Số Model 3

In [4]:
"""
Cell 4: Lọc các biến có ý nghĩa thống kê (P-value < 0.05) ở Model 3
và kiểm tra tính nhất quán logic kinh tế của dấu hệ số.

Kỳ vọng kinh tế:
    - Discount_Ratio: Hệ số > 0 (giảm giá tăng lượng người chơi)
    - Years_Since_Release: Hệ số < 0 (game cũ dần mất người chơi)
"""


def diagnose_coefficient_signs(
    model: RegressionResultsWrapper, model_name: str
) -> pd.DataFrame:
    """Lọc biến có ý nghĩa thống kê và kiểm tra logic kinh tế.

    Args:
        model: Mô hình OLS đã fit.
        model_name: Tên mô hình.

    Returns:
        DataFrame chứa biến có ý nghĩa (p < 0.05) với hệ số và cảnh báo.
    """
    try:
        logger.info("=" * 60)
        logger.info("CHẨN ĐOÁN DẤU HỆ SỐ: %s", model_name)
        logger.info("=" * 60)

        # Lọc biến có P-value < 0.05
        significant: pd.DataFrame = pd.DataFrame({
            "Hệ số": model.params,
            "P-value": model.pvalues,
        })
        significant = significant[significant["P-value"] < 0.05].copy()
        significant["Dấu"] = significant["Hệ số"].apply(
            lambda x: "+" if x > 0 else "-"
        )

        logger.info("Số biến có ý nghĩa thống kê (p < 0.05): %d", len(significant))
        for var_name, row in significant.iterrows():
            logger.info(
                "  %-30s Hệ số = %+.6f  (p = %.6f)",
                var_name, row["Hệ số"], row["P-value"],
            )

        # --- Kiểm tra logic kinh tế ---
        warnings_found: bool = False

        # Discount_Ratio: Kỳ vọng > 0 (giảm giá → tăng người chơi)
        if "Discount_Ratio" in significant.index:
            coeff_dr: float = significant.loc["Discount_Ratio", "Hệ số"]
            if coeff_dr < 0:
                logger.warning(
                    "⚠️ CẢNH BÁO LOGIC KINH TẾ: Discount_Ratio = %.6f < 0. "
                    "Kỳ vọng > 0 (giảm giá phải tăng lượng người chơi). "
                    "Có thể do đa cộng tuyến hoặc endogeneity.",
                    coeff_dr,
                )
                warnings_found = True
            else:
                logger.info(
                    "  ✅ Discount_Ratio = %.6f > 0 — Đúng logic kinh tế", coeff_dr
                )

        # Years_Since_Release: Kỳ vọng < 0 (game cũ mất người chơi)
        if "Years_Since_Release" in significant.index:
            coeff_ysr: float = significant.loc["Years_Since_Release", "Hệ số"]
            if coeff_ysr > 0:
                logger.warning(
                    "⚠️ CẢNH BÁO LOGIC KINH TẾ: Years_Since_Release = %.6f > 0. "
                    "Kỳ vọng < 0 (game cũ dần mất người chơi). "
                    "Có thể do hiệu ứng DLC/Update bù đắp sự lão hóa.",
                    coeff_ysr,
                )
                warnings_found = True
            else:
                logger.info(
                    "  ✅ Years_Since_Release = %.6f < 0 — Đúng logic kinh tế", coeff_ysr
                )

        if not warnings_found:
            logger.info("✅ Tất cả hệ số có ý nghĩa đều nhất quán với logic kinh tế.")

        return significant

    except Exception as e:
        logger.error("LỖI khi chẩn đoán dấu hệ số: %s", e)
        raise


# --- Chạy chẩn đoán cho Model 3 ---
sig_df: pd.DataFrame = diagnose_coefficient_signs(model_3, "Model 3 (Interaction)")

# Hiển thị bảng
sig_df

2026-04-08 22:01:35,522 | INFO | ============================================================


2026-04-08 22:01:35,523 | INFO | CHẨN ĐOÁN DẤU HỆ SỐ: Model 3 (Interaction)


2026-04-08 22:01:35,523 | INFO | ============================================================


2026-04-08 22:01:35,526 | INFO | Số biến có ý nghĩa thống kê (p < 0.05): 6


2026-04-08 22:01:35,527 | INFO |   const                          Hệ số = +0.328570  (p = 0.003383)


...

2026-04-08 22:01:35,530 | INFO |   Is_Weekend                     Hệ số = +0.099460  (p = 0.000000)


2026-04-08 22:01:35,531 | INFO |   Interaction_Discount_Time      Hệ số = -0.130152  (p = 0.002504)


2026-04-08 22:01:35,531 | INFO |   ✅ Discount_Ratio = 0.657613 > 0 — Đúng logic kinh tế


2026-04-08 22:01:35,532 | INFO | ✅ Tất cả hệ số có ý nghĩa đều nhất quán với logic kinh tế.


,Hệ số,P-value,Dấu
const,0.328570,3.382597e-03,+
Lag_Player,0.683308,5.118374e-29,+
Lag_7_Player,0.278262,7.165628e-07,+
Discount_Ratio,0.657613,4.516838e-06,+
Is_Weekend,0.099460,2.154595e-51,+
Interaction_Discount_Time,-0.130152,2.503732e-03,-


---
## Cell 5 — Kiểm Định Out-of-sample (MAPE trên tập Test)

In [5]:
"""
Cell 5: Dùng Model 3 đã train để predict trên tập Test.
Tính chỉ số MAPE (Mean Absolute Percentage Error).

Ngưỡng Production:
    - MAPE < 15%  : Mô hình ĐẠT CHUẨN Production
    - MAPE 15-25% : Chấp nhận được, cần cải thiện
    - MAPE > 25%  : Không đạt, cần refactor
"""


def evaluate_out_of_sample(
    model: RegressionResultsWrapper,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    model_name: str,
) -> float:
    """Đánh giá Out-of-sample bằng MAPE.

    Args:
        model: Mô hình OLS đã fit trên tập Train.
        test_df: Tập Test.
        feature_cols: Danh sách biến độc lập.
        target_col: Tên biến phụ thuộc.
        model_name: Tên mô hình.

    Returns:
        Giá trị MAPE (%).
    """
    try:
        logger.info("=" * 60)
        logger.info("KIỂM ĐỊNH OUT-OF-SAMPLE: %s", model_name)
        logger.info("=" * 60)

        # Chuẩn bị X_test — has_constant='add' bắt buộc thêm cột const
        # để khớp với 11 tham số của mô hình đã train (10 biến + 1 const)
        X_test: pd.DataFrame = sm.add_constant(
            test_df[feature_cols], has_constant='add'
        )
        y_test: pd.Series = test_df[target_col]

        # Dự báo trên tập Test
        y_pred: pd.Series = model.predict(X_test)
        logger.info("Đã predict %d quan sát trên tập Test", len(y_pred))

        # Tính MAPE (tránh chia cho 0)
        mask: pd.Series = y_test != 0
        mape: float = float(
            np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
        )

        logger.info("MAPE = %.4f%%", mape)

        # --- Chẩn đoán tự động ---
        if mape < 15.0:
            logger.info(
                "✅ Mô hình đạt chuẩn Production — MAPE = %.4f%% < 15%%", mape
            )
        elif mape < 25.0:
            logger.warning(
                "🟡 MAPE = %.4f%% — Chấp nhận được nhưng cần cải thiện (15%%-25%%)",
                mape,
            )
        else:
            logger.warning(
                "🔴 MAPE = %.4f%% — KHÔNG ĐẠT chuẩn Production (> 25%%). "
                "Cần refactor mô hình hoặc bổ sung thêm biến.",
                mape,
            )

        # In thêm các chỉ số bổ sung
        mae: float = float(np.mean(np.abs(y_test - y_pred)))
        rmse: float = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))
        logger.info("  MAE  = %.6f", mae)
        logger.info("  RMSE = %.6f", rmse)

        # So sánh thực tế vs dự báo
        comparison: pd.DataFrame = pd.DataFrame({
            "Thực tế (Log_Player)": y_test.values,
            "Dự báo (Predicted)": y_pred.values,
            "Sai số (%)": np.abs((y_test.values - y_pred.values) / y_test.values) * 100,
        }, index=test_df.index)

        return mape

    except Exception as e:
        logger.error("LỖI khi đánh giá Out-of-sample: %s", e)
        raise


# --- Biến độc lập Model 3 ---
model_3_features: list[str] = [
    "Lag_Player", "Lag_7_Player", "Discount_Ratio", "Is_Weekend",
    "Years_Since_Release", "Log_Twitch_Avg", "Trend_Index",
    "Is_Major_Update", "Is_Minor_Update",
    "Interaction_Discount_Time", "Interaction_Update_Time",
]

# --- Chạy kiểm định ---
mape_m3: float = evaluate_out_of_sample(
    model=model_3,
    test_df=test_df,
    feature_cols=model_3_features,
    target_col="Log_Player",
    model_name="Model 3 (Interaction)",
)

logger.info("=" * 60)
logger.info("KẾT THÚC CHẨN ĐOÁN MÔ HÌNH")
logger.info("=" * 60)

2026-04-08 22:01:35,553 | INFO | ============================================================


2026-04-08 22:01:35,554 | INFO | KIỂM ĐỊNH OUT-OF-SAMPLE: Model 3 (Interaction)


2026-04-08 22:01:35,555 | INFO | ============================================================


2026-04-08 22:01:35,561 | INFO | Đã predict 90 quan sát trên tập Test


2026-04-08 22:01:35,564 | INFO | MAPE = 0.5526%


...

2026-04-08 22:01:35,567 | INFO |   MAE  = 0.058426


2026-04-08 22:01:35,568 | INFO |   RMSE = 0.077653


2026-04-08 22:01:35,570 | INFO | ============================================================


2026-04-08 22:01:35,571 | INFO | KẾT THÚC CHẨN ĐOÁN MÔ HÌNH


2026-04-08 22:01:35,573 | INFO | ============================================================
